## Probing: do frozen encodings leak *where the data came from*?

Pathology encoders are notorious for picking up acquisition-site signal -- stain batch, scanner model, institution -- as an easy shortcut, which can hurt generalization even when diagnostic accuracy looks fine in-distribution. This notebook fits linear probes for **site-of-origin** labels at the patch level:

- CAMELYON17 `center` (5-way -- which of the 5 hospitals)
- PANDA `data_provider` (2-way -- Radboud vs. Karolinska)

...and puts the resulting accuracy next to a **diagnostic** probe on the same patches (tumor/normal for CAMELYON17, cancer-present for PANDA) so the two kinds of signal -- "what disease is this" vs. "which hospital scanned this" -- can be compared directly, on the same features.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from nbhelper import (
    plt,
    pd,
    np,
)

from probing import probe_metrics, latest_matching  # noqa: E402

from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODEL_NAMES = ["uni2-h", "virchow2"]

### 1. CAMELYON17: center vs. tumor

Same patch-level features as `02_malignancy_probing_camelyon.ipynb`'s in-distribution split, probed for two different targets.

In [ ]:
camelyon_base_dir = Path("/home/shared/data/camelyon17/camelyon17_v1.0/")
camelyon_encodings_dir = camelyon_base_dir / "encodings"

camelyon_meta_path = latest_matching(camelyon_encodings_dir, "metadata_n*.csv")
camelyon_meta = pd.read_csv(camelyon_meta_path)
camelyon_suffix = camelyon_meta_path.stem.removeprefix("metadata_")
print(
    f"CAMELYON17: {len(camelyon_meta):,} patches, centers={sorted(camelyon_meta['center'].unique())}"
)

# Subsample per center *before* touching the .npy files -- the full release is
# ~450k patches per model, more than needed to fit a quick linear probe.
N_PER_CENTER = 10_000
camelyon_meta = camelyon_meta.groupby("center").sample(n=N_PER_CENTER, random_state=42)
sample_idx = camelyon_meta.index.to_numpy()

camelyon_features = {
    model_name: np.load(
        camelyon_encodings_dir / f"{model_name}_{camelyon_suffix}.npy", mmap_mode="r"
    )[sample_idx]
    for model_name in MODEL_NAMES
}
print(f"CAMELYON17: subsampled to {len(camelyon_meta):,} patches")

### 2. PANDA: data provider vs. cancer-present

`00_preparation/01_encodings/encode_panda.ipynb` never writes one combined per-patch feature array -- across ~10k slides it wouldn't fit in memory, so patch-level features only exist scattered across per-slide `encodings/patches/{image_id}.npz` files (each holding that slide's own `x0`, `y0`, and per-model patch-token arrays). `patch_metadata_all.csv` has the patch-level labels (`isup_grade`, `data_provider`) but no features, so this section subsamples at the *slide* level first, then loads just those slides' `.npz` files to pull out their patch features -- every sampled patch inherits its parent slide's `isup_grade`/`data_provider`, since PANDA has no per-patch annotation. `isup_grade == 0` (benign) vs. `> 0` (any cancer) is a coarse but patch-applicable diagnostic label.</cell id="a3c2f96b-43e0-443b-a0a4-76c2d16b4ec3">


In [ ]:
panda_base_dir = Path("/home/shared/data/panda/")
panda_encodings_dir = Path("/home/shared/data/panda/encodings")
panda_patches_dir = panda_encodings_dir / "patches"  # per-slide .npz files -- features live here

panda_meta_path = latest_matching(panda_encodings_dir, "patch_metadata_*.csv")
panda_meta_all = pd.read_csv(panda_meta_path)
print(
    f"PANDA: {len(panda_meta_all):,} patches across {panda_meta_all['image_id'].nunique():,} "
    f"slides, providers={sorted(panda_meta_all['data_provider'].unique())}"
)

# Subsample *slides*, not patches -- there's no global per-patch feature array to
# index into, so instead pick a handful of slides per provider and load just
# their .npz files, each of which holds that slide's own patch features.
N_SLIDES_PER_PROVIDER = 15
slide_info = panda_meta_all.drop_duplicates("image_id").set_index("image_id")
sampled_slides = slide_info.groupby("data_provider").sample(
    n=N_SLIDES_PER_PROVIDER, random_state=42
)

panda_meta_rows = []
panda_features_lists = {model_name: [] for model_name in MODEL_NAMES}
for image_id, row in sampled_slides.iterrows():
    npz = np.load(panda_patches_dir / f"{image_id}.npz")
    n_patches = len(npz["x0"])
    panda_meta_rows.append(
        pd.DataFrame(
            {
                "image_id": image_id,
                "isup_grade": row["isup_grade"],
                "data_provider": row["data_provider"],
                "cancer_present": int(row["isup_grade"] > 0),
            },
            index=range(n_patches),
        )
    )
    for model_name in MODEL_NAMES:
        panda_features_lists[model_name].append(npz[model_name])

panda_meta = pd.concat(panda_meta_rows, ignore_index=True)
panda_features = {
    model_name: np.concatenate(panda_features_lists[model_name]) for model_name in MODEL_NAMES
}
print(
    f"PANDA: subsampled to {len(panda_meta):,} patches from {len(sampled_slides)} slides "
    f"({N_SLIDES_PER_PROVIDER} per provider)"
)

In [ ]:
from tqdm.auto import tqdm


def fit_linear_probe(X_train, y_train, X_test, y_test, labels=None, max_iter=2000, seed=42):
    """Standardize + multinomial logistic regression -- the standard frozen-
    feature ("linear probe") evaluation for self-supervised encoders."""
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=max_iter, random_state=seed))
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return clf, probe_metrics(y_test, y_pred, labels=labels)


def probe_target(X, y, labels):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    _, metrics = fit_linear_probe(X_train, y_train, X_test, y_test, labels=labels)
    return metrics["accuracy"], metrics["macro_f1"]


# .to_numpy() rather than .values -- data_provider is a string column and, on
# this pandas/pyarrow setup, .values on it returns the ArrowExtensionArray as-is
# instead of a plain ndarray, which sklearn's train_test_split can't fancy-index.
tasks = [
    (
        "CAMELYON17",
        "tumor (diagnostic)",
        camelyon_features,
        camelyon_meta["tumor"].to_numpy(),
        [0, 1],
    ),
    (
        "CAMELYON17",
        "center (site)",
        camelyon_features,
        camelyon_meta["center"].to_numpy(),
        sorted(camelyon_meta["center"].unique()),
    ),
    (
        "PANDA",
        "cancer-present (diagnostic)",
        panda_features,
        panda_meta["cancer_present"].to_numpy(),
        [0, 1],
    ),
    (
        "PANDA",
        "data_provider (site)",
        panda_features,
        panda_meta["data_provider"].to_numpy(),
        sorted(panda_meta["data_provider"].unique()),
    ),
]

rows = []
for dataset, task_name, feats, y, labels in tqdm(tasks):
    chance = 1.0 / len(labels)
    for model_name in tqdm(MODEL_NAMES, leave=False):
        acc, macro_f1 = probe_target(feats[model_name], y, labels)
        rows.append(
            {
                "dataset": dataset,
                "task": task_name,
                "encoder": model_name,
                "n_classes": len(labels),
                "chance_accuracy": chance,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "accuracy_over_chance": acc - chance,
            }
        )

leak_df = pd.DataFrame(rows)
leak_df

### 4. Diagnostic signal vs. site signal, above chance

Comparing raw accuracy across a 2-way and a 5-way probe is misleading -- accuracy *above chance* puts them on the same footing.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
task_order = [
    "tumor (diagnostic)",
    "center (site)",
    "cancer-present (diagnostic)",
    "data_provider (site)",
]
x = np.arange(len(task_order))
width = 0.35

for i, model_name in enumerate(MODEL_NAMES):
    sub = leak_df[leak_df["encoder"] == model_name].set_index("task").loc[task_order]
    ax.bar(x + i * width, sub["accuracy_over_chance"], width, label=model_name)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(task_order, rotation=20, ha="right")
ax.set_ylabel("linear-probe accuracy - chance")
ax.axvline(1.5, color="gray", linestyle=":", alpha=0.6)
ax.text(0.75, ax.get_ylim()[1] * 0.95, "CAMELYON17", ha="center", fontsize=9)
ax.text(2.75, ax.get_ylim()[1] * 0.95, "PANDA", ha="center", fontsize=9)
ax.legend()
ax.set_title("diagnostic vs. site-of-origin: how much each is linearly decodable")
plt.tight_layout()
plt.show()

### Takeaways

- If the site-of-origin bars are comparable to (or taller than) the diagnostic bars, acquisition site is at least as linearly readable as the label you actually care about -- a real shortcut-learning risk, and a reason to check `02_malignancy_probing_camelyon.ipynb`'s cross-center results carefully rather than trusting in-distribution accuracy alone.
- This is descriptive, not a fix -- it doesn't say the diagnostic probe is *using* the site shortcut, only that the shortcut is available in the embedding space for a probe to find.
- Whichever encoder shows a smaller site-signal bar (relative to its diagnostic-signal bar) is arguably the more "domain-invariant" of the two for these datasets.